In [ ]:
import pandas as pd
from pathlib import Path
df = pd.read_parquet('predictions_01_11_2022.parquet')
df = df.fillna('')
df

In [ ]:
# Reorder columns
order = ['comp', 'Tg', 'Tm',  'YM', 'TS_b',  'eps_b', 'perm_O2', 'perm_CO2', 'Td', 'TS_y', 'perm_N2',  
       'perm_H2',  'perm_He',   'perm_CH4', 'smiles', 'num_side', 'num_back', 'end_group', 'names']
df = df[order]
df

In [ ]:
df_exp = pd.read_pickle('exp_values.pkl')

# PLA properties from published literature
# Each value traced to a specific peer-reviewed source
pla_row = pd.DataFrame({
    'Polymer': ['Poly(lactic acid)'],
    'Abb.': ['PLA'],
    'applications': ['Packaging, 3D printing, medical implants'],
    'smiles': ['[*]OC(C)C(=O)[*]'],
    '%-usage': [0],
    'Tg': [333],       # Avérous 2008: reports 323-353 K range; 333 K is midpoint for typical commercial PLA
    'Tm': [443],        # Farah et al. 2016, Adv. Drug Deliv. Rev.: Table of physical properties of high Mw PLA
    'TS_b': [60.0],     # Santos et al. 2015, Procedia Eng. 114: 62.7 MPa for neat PLA (NatureWorks, 170k g/mol); rounded to 60
    'eps_b': [6.0],     # MakeItFrom.com (citing Auras et al. 2010, PLA textbook); also Wikipedia: "less than 10%", 6% for pure PLLA
    'YM': [1900],       # Farah et al. 2016, Adv. Drug Deliv. Rev.: comparison table of PLA physical properties
    'perm_O2': [0.26],  # Bao, Dorgan et al. 2006, J. Membr. Sci. 285, 166-172: time-lag method, PLA 98.7% L at 30°C
    'perm_CO2': [1.10],  # Bao, Dorgan et al. 2006, J. Membr. Sci. 285, 166-172: time-lag method, PLA 98.7% L at 30°C
    'source': ['Tg: Avérous 2008; Tm: Farah et al. 2016; TS_b: Santos et al. 2015; eps_b: Auras et al. 2010; YM: Farah et al. 2016; perm_O2/CO2: Bao & Dorgan 2006']
})

df_exp = pd.concat([df_exp, pla_row], ignore_index=True)
df_exp

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# props to match
search_props = ['Tg', 'Tm', 'TS_b', 'YM',  'eps_b', 'perm_O2', 'perm_CO2']

# weights 
weights = np.array([1, 1, 1, 1, 1, 1, 1])
params = dict(
    metric='minkowski',
    p=2
)

# PHAs only
df_pha_only = df[df.names.apply(lambda x: list(x) == ['pha', 'pha'])]
nn_pha = NearestNeighbors(**params).fit(df_pha_only[search_props])

# All others (with conventional polymers)
df_pha_conv = df[df.names.apply(lambda x: list(x) != ['pha', 'pha'])]
nn_conv = NearestNeighbors(**params).fit(df_pha_conv[search_props])


# Nearest Neighbor for phas phas

In [ ]:

n_neighbors = 5 
results = []
for n, row in df_exp.iterrows():
    r = row[search_props]
    dist, idx = nn_pha.kneighbors(r.to_frame().T, n_neighbors)
    res = df_pha_only.iloc[idx[0]].copy()
    res.insert(0, 'polymer', row.Polymer)
    res.insert(1, 'abb', row['Abb.'])
    res.insert(1, 'type', 'pha_only')

    diff = res[search_props] - r.to_frame().T.values
    diff = diff.add_prefix('d_')
    res = pd.concat([res, diff], axis=1)
    results.append(res)

    dist, idx = nn_conv.kneighbors(r.to_frame().T, n_neighbors)
    res = df_pha_conv.iloc[idx[0]].copy()
    res.insert(0, 'polymer', row.Polymer)
    res.insert(1, 'abb', row['Abb.'])
    res.insert(2, 'type', 'pha_conv')

    diff = res[search_props] - r.to_frame().T.values
    diff = diff.add_prefix('d_')
    res = pd.concat([res, diff], axis=1)

    results.append(res)

df_res = pd.concat(results).set_index(['polymer', 'type'])  

df_res.insert(0, 'smiles', df_res.pop('smiles'))
df_res.insert(0, 'names', df_res.pop('names'))

pp = Path('export')
pp.mkdir(exist_ok=True)
df_res.to_csv(pp / 'candidates.csv')
df_res[df_res.abb == 'PLA']

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

import numpy as np

for name, gr in df_res.reset_index().groupby(by="abb", sort=False):
    sms = np.array([[x[0], x[1]] for x in gr.smiles.values])
    mols = [Chem.MolFromSmiles(x) for x in sms.flatten()]
    comps = gr.comp.values

    for num, sm in enumerate(sms):
        sms[num][0] = f"{round(comps[num],2)}: {sms[num][0]}"
    nn = name.lower().replace(" ", "")
    img = Draw.MolsToGridImage(
        mols,
        subImgSize=(200, 200),
        molsPerRow=2,
        maxMols=200,
        useSVG=True,
        legends=sms.flatten().tolist(),
    )
    Path(pp / f"{nn}.svg").write_text(img.data)
